In [ ]:
import os
from os.path import join
from time import time
from itertools import combinations

from joblib import Parallel, delayed
import numpy as np

from ase.io import read, write
from ase.visualize import view
from ase.build import fcc100, fcc111, fcc211, fcc110
from ase.constraints import FixAtoms

from mylib.theory import theo_kpoints
from mylib.slabs import create_surface, level_correction, invert_selective_dynamics_new, fix_positions
from mylib.fixers import sort_atoms_by_symbol

In [ ]:
def get_combos(slab, n_sub):
    tags = slab.get_tags()
    mask = (tags==5) + (tags==6)
    possible_indices = list(np.argwhere(mask).flatten())
    chosen_indices = [0] 
    possible_indices = [i for i in possible_indices if i not in chosen_indices]

    sub_combos = (list(combinations(possible_indices, n_sub-len(chosen_indices))))
    return [[*chosen_indices, *sub_combo] for sub_combo in sub_combos]


In [ ]:
def get_mean_distance_of_combo(all_distances, combo):
    sub = all_distances[np.ix_(combo, combo)]
    N = len(sub)
    return np.mean(sub)*(1-N/N**2)


def get_min_distance_of_combo(all_distances, combo):
    sub = all_distances[np.ix_(combo, combo)]
    N = len(sub)
    return np.mean(sub)*(1-N/N**2)

In [ ]:

def find_ideal_combo(distances, all_combos, n_jobs=-1):
    avg_distances = Parallel(n_jobs=n_jobs)(
        delayed(get_mean_distance_of_combo)(distances, combo) for combo in all_combos
    )
    M = np.max(avg_distances)
    avg_indices = np.where(avg_distances == M)[0]

    min_distances = Parallel(n_jobs=n_jobs)(
        delayed(get_min_distance_of_combo)(distances, combo) for combo in all_combos
    )
    M = np.max(min_distances)
    min_indices = np.where(min_distances == M)[0]

    return np.intersect1d(avg_indices, min_indices)[0]


In [ ]:
def get_all_indices_to_be_replaced(all_positions, combo):
    final_indices_to_replace = np.all(
        all_positions[:, :2] == all_positions[0, :2], axis=1
    )
    for index in combo[1:]:
        final_indices_to_replace += np.all(
            all_positions[:, :2] == all_positions[index, :2], axis=1
        )
    return final_indices_to_replace

In [ ]:
def replace_copper_with_gallia(slab_original, indices):
    new_slab = slab_original.copy()
    new_atomic_numbers = np.ones_like(indices)*29+2*indices
    new_slab.set_atomic_numbers(new_atomic_numbers)
    return new_slab

In [ ]:
def extract_lattice(path):
    atoms = read(path, format = "vasp")
    return atoms.get_cell()[0,0]/2

In [ ]:
slab = fcc111("Cu", size = (4,4,6),a = 3.14, vacuum=7.5 )
view(slab)



In [ ]:
for n_sub in range(1,7):
    
    alpha = extract_lattice(join("homo-CONTCAR", str(n_sub)))
    
    print(n_sub, alpha)
    slab = fcc111("Cu", size = (4,4,6),a = alpha, vacuum=7.5 )
    all_positions = slab.get_positions()
    distances = slab.get_all_distances(mic = True)
    combos = get_combos(slab, n_sub)
    i = find_ideal_combo(distances, combos)
    # print(i)
    all_i = get_all_indices_to_be_replaced(all_positions, combos[i])
    # print(all_i)
    new_slab = replace_copper_with_gallia(slab, all_i)
    a_slab = fix_positions(new_slab.copy()) 
    b_slab = invert_selective_dynamics_new(a_slab.copy()) 
    
    # view(new_slab)
    # view(a_slab)
    # view(b_slab)
    os.mkdir(join("./homo-surfaces/", f"{n_sub}-111"))
    os.mkdir(join("./homo-surfaces/", f"{n_sub}-111-a"))
    os.mkdir(join("./homo-surfaces/", f"{n_sub}-111-b"))
    
    
    
    write(join("./homo-surfaces/", f"{n_sub}-111", "POSCAR"), new_slab, format = "vasp", sort=True)
    write(join("./homo-surfaces/", f"{n_sub}-111-a", "POSCAR"), a_slab, format = "vasp", sort=True)
    write(join("./homo-surfaces/", f"{n_sub}-111-b", "POSCAR"), b_slab, format = "vasp", sort=True)
    
    

In [ ]:
for file in os.listdir("./homo-surfaces/"):
    con,miller, *termination = file.split("-")
    new_termination = 0 if len(termination)==0 else termination[0]
    new_name = f"{con}-110-{new_termination}"
    
    os.rename(join("./homo-surfaces/", file), join("./homo-surfaces/", new_name))
    
